# Starbucks Location Fitness Score: Predictive Model

**Predicting store-location fitness from spatial, transit, and competition features in Manhattan.**

This notebook builds a regression model to predict the *Location Fitness Score* (LFS) — a composite metric that captures how well a Starbucks location is positioned relative to demand drivers and competitive supply. We compare linear, random-forest, and gradient-boosting regressors, analyse feature importance, and export the best model for the [Kaggle Models](https://www.kaggle.com/models) hub.

**Series context**

| # | Notebook | Link |
|---|----------|------|
| 0 | Manhattan Café Wars — EDA | [Kaggle](https://www.kaggle.com/code/shiratoriseto/manhattan-cafe-wars-starbucks-vs-1200-competitors) |
| 1 | Starbucks 10-K NLP | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-nlp-topic-keyword-analysis) |
| 2A | Spatial Clustering | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-spatial-clustering) |
| 2B | Location Fitness | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-location-fitness) |
| **5** | **LFS Predictive Model (this notebook)** | — |

## Section 0 — Setup & Data Loading

In [ ]:
# ── imports ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import json
from pathlib import Path

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})

# ── data path resolution ────────────────────────────────────────────
ON_KAGGLE = Path("/kaggle/working").exists()

if ON_KAGGLE:
    _candidates = [
        Path("/kaggle/input/manhattan-cafe-wars"),
        Path("/kaggle/input/datasets/shiratoriseto/manhattan-cafe-wars"),
    ]
    DATA_DIR = None
    for _p in _candidates:
        if _p.exists():
            DATA_DIR = _p
            break
    if DATA_DIR is None:
        import os
        avail = os.listdir("/kaggle/input") if Path("/kaggle/input").exists() else []
        raise FileNotFoundError(f"Dataset not found. Available: {avail}")
else:
    DATA_DIR = Path("../../dataset-upload/v3")

# ── load data ────────────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / "stores_enriched_v4.csv")
print(f"Loaded {df.shape[0]} rows × {df.shape[1]} cols")
print(f"Target stats — min: {df['location_fitness_score'].min():.3f}, "
      f"median: {df['location_fitness_score'].median():.3f}, "
      f"max: {df['location_fitness_score'].max():.3f}")
df.head(3)

## Section 1 — Feature Engineering

In [ ]:
# ── feature selection ────────────────────────────────────────────────
# IMPORTANT: demand_proxy_index and supply_index are EXCLUDED.
# LFS = demand_proxy / supply, so using them would be data leakage.

feature_cols = [
    # Building
    "pluto_numfloors", "pluto_yearbuilt", "pluto_retailarea",
    "pluto_comarea", "pluto_lotarea", "pluto_assesstot",
    # Transit
    "mta_dist_m", "mta_avg_daily_ridership",
    # Competition
    "n_starbucks_500m", "n_dunkin_500m", "n_other_cafe_500m",
    "nearest_competitor_dist_m", "nearest_starbucks_dist_m",
    # Census
    "tract_population", "tract_median_income",
    "tract_pct_walk_commute", "tract_pct_bachelors_plus",
    # Pedestrian
    "ped_count_nearest",
    # Location
    "lat", "lon",
]

TARGET = "location_fitness_score"

# Drop rows with NaN in features or target
cols_needed = feature_cols + [TARGET]
df_model = df[cols_needed].dropna().copy()
print(f"After dropping NaNs: {df_model.shape[0]} rows × {df_model.shape[1]} cols "
      f"(dropped {len(df) - len(df_model)} rows)")

# ── correlation with target ──────────────────────────────────────────
corr_with_target = (
    df_model[feature_cols]
    .corrwith(df_model[TARGET])
    .sort_values(key=abs, ascending=False)
)
print("\nTop feature correlations with LFS:")
print(corr_with_target.to_string())

## Section 2 — Baseline & Model Comparison

In [ ]:
# ── train / test split ───────────────────────────────────────────────
X = df_model[feature_cols].values
y = df_model[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")

# ── model comparison via 5-fold CV ───────────────────────────────────
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=200, random_state=42),
}

results = []
for name, model in models.items():
    cv_mae = -cross_val_score(model, X_train, y_train,
                              cv=5, scoring="neg_mean_absolute_error")
    cv_r2 = cross_val_score(model, X_train, y_train,
                            cv=5, scoring="r2")
    results.append({
        "Model": name,
        "CV MAE (mean)": f"{cv_mae.mean():.3f}",
        "CV MAE (std)": f"{cv_mae.std():.3f}",
        "CV R² (mean)": f"{cv_r2.mean():.3f}",
        "CV R² (std)": f"{cv_r2.std():.3f}",
    })

results_df = pd.DataFrame(results)
print("\n5-Fold Cross-Validation Results (on training set):")
print(results_df.to_string(index=False))

## Section 3 — Best Model: Feature Importance

In [ ]:
# ── select best model and retrain on full training set ───────────────
best_model = GradientBoostingRegressor(n_estimators=200, random_state=42)
best_model.fit(X_train, y_train)
print(f"Best model: {type(best_model).__name__}")

# ── built-in feature importance ─────────────────────────────────────
fi = pd.Series(best_model.feature_importances_, index=feature_cols)
fi_sorted = fi.sort_values(ascending=True)

# ── permutation importance on test set ──────────────────────────────
perm = permutation_importance(best_model, X_test, y_test,
                              n_repeats=20, random_state=42)
pi = pd.Series(perm.importances_mean, index=feature_cols)
pi_sorted = pi.sort_values(ascending=True)

# ── side-by-side plots ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].barh(fi_sorted.index, fi_sorted.values, color="#00704A")
axes[0].set_title("Built-in Feature Importance (Gini)")
axes[0].set_xlabel("Importance")

axes[1].barh(pi_sorted.index, pi_sorted.values, color="#1E3932")
axes[1].set_title("Permutation Importance (Test Set)")
axes[1].set_xlabel("Mean decrease in R²")

plt.tight_layout()
plt.show()

## Section 4 — Model Evaluation

In [ ]:
# ── predictions on test set ─────────────────────────────────────────
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Test MAE:  {mae:.4f}")
print(f"Test RMSE: {rmse:.4f}")
print(f"Test R²:   {r2:.4f}")

# ── evaluation plots ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 1. Actual vs Predicted
ax = axes[0]
ax.scatter(y_test, y_pred, alpha=0.6, edgecolors="k", linewidth=0.5,
           color="#00704A", s=50)
lo = min(y_test.min(), y_pred.min()) - 0.1
hi = max(y_test.max(), y_pred.max()) + 0.1
ax.plot([lo, hi], [lo, hi], "--", color="grey", linewidth=1)
ax.set_xlabel("Actual LFS")
ax.set_ylabel("Predicted LFS")
ax.set_title(f"Actual vs Predicted  (R² = {r2:.3f})")
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_aspect("equal")

# 2. Residual distribution
residuals = y_test - y_pred
ax = axes[1]
ax.hist(residuals, bins=20, color="#1E3932", edgecolor="white")
ax.axvline(0, color="red", linestyle="--", linewidth=1)
ax.set_xlabel("Residual (Actual − Predicted)")
ax.set_ylabel("Count")
ax.set_title("Residual Distribution")

plt.tight_layout()
plt.show()

## Section 5 — Spatial Residual Analysis

In [ ]:
# ── spatial residuals ────────────────────────────────────────────────
# Recover lat/lon for test rows
lat_idx = feature_cols.index("lat")
lon_idx = feature_cols.index("lon")
test_lat = X_test[:, lat_idx]
test_lon = X_test[:, lon_idx]

fig, ax = plt.subplots(figsize=(8, 10))
sc = ax.scatter(test_lon, test_lat, c=residuals, cmap="RdYlGn",
                edgecolors="k", linewidth=0.5, s=70, vmin=-1, vmax=1)
plt.colorbar(sc, ax=ax, label="Residual (Actual − Predicted)",
             shrink=0.7)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Spatial Residuals — Green = under-predicted, Red = over-predicted")
plt.tight_layout()
plt.show()

# ── worst predictions ────────────────────────────────────────────────
abs_resid = np.abs(residuals)
worst_idx = np.argsort(abs_resid)[-5:][::-1]

print("\nTop 5 worst predictions:")
print(f"{'Idx':>4}  {'Actual':>8}  {'Predicted':>10}  {'Residual':>9}  {'Lat':>9}  {'Lon':>10}")
for i in worst_idx:
    print(f"{i:4d}  {y_test[i]:8.3f}  {y_pred[i]:10.3f}  {residuals[i]:9.3f}  "
          f"{test_lat[i]:9.5f}  {test_lon[i]:10.5f}")

## Section 6 — Export Model

In [ ]:
# ── save model & metadata ────────────────────────────────────────────
import joblib, json

model_path = Path("/kaggle/working" if ON_KAGGLE else ".") / "lfs_model.joblib"
joblib.dump(best_model, model_path)

metadata = {
    "model_type": type(best_model).__name__,
    "features": feature_cols,
    "target": "location_fitness_score",
    "n_train": len(X_train),
    "n_test": len(X_test),
    "test_mae": float(mae),
    "test_r2": float(r2),
    "description": "Predicts Starbucks location fitness score from spatial, transit, and competition features in Manhattan"
}
meta_path = Path("/kaggle/working" if ON_KAGGLE else ".") / "lfs_model_metadata.json"
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Model saved: {model_path} ({model_path.stat().st_size / 1024:.0f} KB)")
print(f"Metadata saved: {meta_path}")
print(f"\nModel summary:")
print(f"  Type:     {metadata['model_type']}")
print(f"  Features: {len(metadata['features'])}")
print(f"  Test MAE: {metadata['test_mae']:.4f}")
print(f"  Test R²:  {metadata['test_r2']:.4f}")

## Section 7 — Limitations & Series Navigation

### Limitations

| # | Limitation | Impact |
|---|-----------|--------|
| 1 | **Small dataset** (n=171) limits model complexity | Cannot fit deep ensembles or neural nets without severe overfitting |
| 2 | **Single time snapshot** — no temporal validation | Model may not capture seasonal or trend effects |
| 3 | **LFS target is a derived metric**, not ground truth (e.g., revenue) | Prediction accuracy is bounded by the quality of the LFS formula |
| 4 | **Manhattan-specific** — may not generalise to other cities | Feature distributions (transit density, rent) differ dramatically elsewhere |
| 5 | **No hyperparameter tuning** (intentionally simple) | Kept simple for interpretability; tuning could improve R² modestly |

### Series Navigation

| # | Notebook | Link |
|---|----------|------|
| 0 | Manhattan Café Wars — EDA | [Kaggle](https://www.kaggle.com/code/shiratoriseto/manhattan-cafe-wars-starbucks-vs-1200-competitors) |
| 1 | Starbucks 10-K NLP: Topic & Keyword Analysis | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-nlp-topic-keyword-analysis) |
| 1F | Starbucks 10-K LDA Topic Explorer (pyLDAvis) | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-lda-topic-explorer-pyldavis) |
| 2A | Starbucks Spatial Clustering | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-spatial-clustering) |
| 2B | Starbucks Location Fitness | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-location-fitness) |
| 2C | Starbucks Walk-Distance Analysis (OSMnx) | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-walk-distance-analysis-osmnx) |
| **5** | **LFS Predictive Model (this notebook)** | — |
| C | Data Pipeline: EDGAR + OSM → CSV | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-data-pipeline-edgar-osm-to-csv) |
| D | US Expansion Animated Choropleth | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-us-expansion-animated-choropleth) |
| G | NLP × Spatial Integration | [Kaggle](https://www.kaggle.com/code/shiratoriseto/starbucks-nlp-x-spatial) |

**Dataset:** [shiratoriseto/manhattan-cafe-wars](https://www.kaggle.com/datasets/shiratoriseto/manhattan-cafe-wars)